# [SOLUTION] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [37]:
# Only needed for Udacity workspace
import importlib.util
import sys

if importlib.util.find_spec('pysqlite3') is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [38]:
import os
import json
from dataclasses import dataclass
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [ ]:
# TODO: Create a .env file with the following variables
# OPENAI_API_KEY="YOUR_KEY"
# CHROMA_OPENAI_API_KEY="YOUR_KEY"
# TAVILY_API_KEY="YOUR_KEY"

In [39]:
# Load environment variables
load_dotenv()

False

### VectorDB Instance

In [40]:
# Instantiate your ChromaDB Client
chroma_client = chromadb.PersistentClient(path='chromadb')
print('Chroma client created')

Chroma client created


### Collection

In [41]:
# OpenAI embedding function — must use the same model when querying
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key    = os.getenv("OPENAI_KEY"),
    api_base   = os.getenv("OPENAI_BASE"),
    model_name = 'text-embedding-ada-002',
)

In [42]:
# Drop any existing collection and create a fresh one
try:
    chroma_client.delete_collection('udaplay')
    print('Deleted existing collection')
except Exception:
    pass

collection = chroma_client.create_collection(
    name = 'udaplay',
    embedding_function = embedding_fn,
)
print(f"Collection '{collection.name}' created")

Deleted existing collection
Collection 'udaplay' created


### Add documents

In [44]:
# Make sure you have a directory 'games'
data_dir = 'games'

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith('.json'):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, 'r', encoding='utf-8') as f:
        game = json.load(f)

    # Rich text representation — captures all searchable dimensions
    content = (
        f"Title: {game['Name']}\n"
        f"Platform: {game['Platform']} | Year: {game['YearOfRelease']} | Genre: {game['Genre']}\n"
        f"Publisher: {game['Publisher']}\n"
        f"About: {game['Description']}"
    )

    doc_id = os.path.splitext(file_name)[0]   # filename '001'

    collection.add(
        ids       = [doc_id],
        documents = [content],
        metadatas = [game],
    )
    print(f'  {doc_id} - {game["Name"]}')

print(f'\nTotal documents in collection: {collection.count()}')

  001 - Gran Turismo
  002 - Grand Theft Auto: San Andreas
  003 - Gran Turismo 5
  004 - Marvel's Spider-Man
  005 - Marvel's Spider-Man 2
  006 - Pokémon Gold and Silver
  007 - Pokémon Ruby and Sapphire
  008 - Super Mario World
  009 - Super Mario 64
  010 - Super Smash Bros. Melee
  011 - Wii Sports
  012 - Mario Kart 8 Deluxe
  013 - Kinect Adventures!
  014 - Minecraft
  015 - Halo Infinite

Total documents in collection: 15


### Semantic Search Demo

Validate that the collection returns sensible results for varied query styles.

In [45]:
test_queries = [
    'The sequel to the acclaimed Spider-Man game?',
    'Publisher of Game Boy Color?',
    'Grand Theft Auto: San Andreas released year?'
]

for q in test_queries:
    results = collection.query(
        query_texts = [q],
        n_results   = 2,
        include     = ['metadatas'],
    )
    print(f'\nQuery: "{q}"')
    print(f'Results: "{results}"')
    for meta in results['metadatas'][0]:
        print(f'  → {meta["Name"]} - ({meta["Platform"]} - {meta["YearOfRelease"]})')


Query: "The sequel to the acclaimed Spider-Man game?"
Results: "{'ids': [['005', '004']], 'embeddings': None, 'documents': None, 'uris': None, 'included': ['metadatas'], 'data': None, 'metadatas': [[{'YearOfRelease': 2023, 'Platform': 'PlayStation 5', 'Description': 'The sequel to the acclaimed Spider-Man game, featuring both Peter Parker and Miles Morales as playable characters.', 'Genre': 'Action-adventure', 'Name': "Marvel's Spider-Man 2", 'Publisher': 'Sony Interactive Entertainment'}, {'Description': 'An open-world superhero game that lets players swing through New York City as Spider-Man, battling iconic villains.', 'Genre': 'Action-adventure', 'Publisher': 'Sony Interactive Entertainment', 'Name': "Marvel's Spider-Man", 'YearOfRelease': 2018, 'Platform': 'PlayStation 4'}]], 'distances': None}"
  → Marvel's Spider-Man 2 - (PlayStation 5 - 2023)
  → Marvel's Spider-Man - (PlayStation 4 - 2018)

Query: "Publisher of Game Boy Color?"
Results: "{'ids': [['006', '007']], 'embeddings'